In [1]:
!pip install transformers torch scikit-learn arabert -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 6.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from arabert.preprocess import ArabertPreprocessor

In [15]:
MODEL_NAME = "aubmindlab/bert-base-arabertv2"
MAX_LEN = 128
EPOCHS = 3
BATCH = 8
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [5]:
from google.colab import files
uploaded = files.upload()

train_df = pd.read_csv("train.csv", encoding="utf-8-sig")
val_df   = pd.read_csv("validation.csv", encoding="utf-8-sig")
test_df  = pd.read_csv("test.csv", encoding="utf-8-sig")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(train_df["label"].value_counts())

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving validation.csv to validation.csv
Train: 6000 | Val: 1108 | Test: 3000
label
1    3045
0    2955
Name: count, dtype: int64


In [7]:
def clean_arabic(text):
    text = re.sub(r'[\u064B-\u065F]', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s\d\.,!?؟،]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

preprocessor = ArabertPreprocessor(model_name=MODEL_NAME)

for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].apply(clean_arabic)
    df["text"] = df["text"].apply(preprocessor.preprocess)

print("تم")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'farasa-api.qcri.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



  0%|          | 0.00/241M [00:00<?, ?iB/s]
  1%|▏         | 3.41M/241M [00:05<06:45, 586kiB/s]
  3%|▎         | 6.83M/241M [00:10<06:01, 649kiB/s]
  4%|▍         | 10.2M/241M [00:18<07:10, 537kiB/s]
  6%|▌         | 13.7M/241M [00:25<07:13, 525kiB/s]
  7%|▋         | 17.1M/241M [00:31<06:59, 535kiB/s]
  8%|▊         | 20.5M/241M [00:36<06:28, 569kiB/s]
 10%|▉         | 23.9M/241M [00:42<06:14, 581kiB/s]
 11%|█▏        | 27.3M/241M [00:47<05:56, 601kiB/s]
 13%|█▎        | 30.7M/241M [00:52<05:41, 616kiB/s]
 14%|█▍        | 34.1M/241M [00:59<06:02, 572kiB/s]
 16%|█▌        | 37.5M/241M [01:04<05:42, 594kiB/s]
 17%|█▋        | 41.0M/241M [01:12<06:08, 544kiB/s]
 18%|█▊        | 44.4M/241M [01:18<06:06, 537kiB/s]
 20%|█▉        | 47.8M/241M [01:25<06:00, 537kiB/s]
 21%|██        | 51.2M/241M [01:32<06:09, 514kiB/s]
 23%|██▎       | 54.6M/241M [01:39<06:13, 500kiB/s]
 24%|██▍       | 58.0M/241M [01:45<05:44, 532kiB/s]
 25%|██▌       | 61.4M/241M [01:51<05:34, 537kiB/s]
 27%|██▋       | 64

[2026-04-07 06:34:44,768 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


تم


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_enc = tokenize(train_df["text"])
val_enc   = tokenize(val_df["text"])
test_enc  = tokenize(test_df["text"])
print(" تم")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

 تم


In [9]:
class ArabicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = ArabicDataset(train_enc, list(train_df["label"]))
val_dataset   = ArabicDataset(val_enc,   list(val_df["label"]))
test_dataset  = ArabicDataset(test_enc,  list(test_df["label"]))
print(" تم")

 تم


In [10]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor.to(model.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print("تم ")

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

تم 


In [16]:
from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.049781,0.241984,0.968412,0.968410
2,0.026400,0.198910,0.971119,0.971120
3,0.016480,0.192441,0.972924,0.972923


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2250, training_loss=0.026792806095547145, metrics={'train_runtime': 8730.1277, 'train_samples_per_second': 2.062, 'train_steps_per_second': 0.258, 'total_flos': 351499925520000.0, 'train_loss': 0.026792806095547145, 'epoch': 3.0})

In [17]:
results = trainer.predict(test_dataset)
preds   = np.argmax(results.predictions, axis=-1)
y_test  = np.array(test_df["label"])

print(classification_report(y_test, preds, target_names=["غير تشبيه", "تشبيه"]))

accs = []
for _ in range(1000):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    accs.append(accuracy_score(y_test[idx], preds[idx]))

print(f"Bootstrap Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

   غير تشبيه       0.97      0.96      0.97      1478
       تشبيه       0.96      0.97      0.97      1522

    accuracy                           0.97      3000
   macro avg       0.97      0.97      0.97      3000
weighted avg       0.97      0.97      0.97      3000

Bootstrap Accuracy: 0.9669 ± 0.0033


In [18]:
import shutil
model.save_pretrained("./my-arabert-model")
tokenizer.save_pretrained("./my-arabert-model")
shutil.make_archive("My_AraBERT_Model", "zip", "./my-arabert-model")

from google.colab import files
files.download("My_AraBERT_Model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>